In [1]:
# import os
# os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

## Importing libraries

In [1]:
from beir.datasets.data_loader import GenericDataLoader
from helpers.converter import convert_corpus_jsonl
from sparse.bm25 import BM25Retrieval
from sparse.splade import Splade
import time
import os
from evaluation.metrics import compute_ndcg
from sentence_transformers import SentenceTransformer
from dense.dense_retrievers import HNSW, FlatIndex
from models.bge import embed_model
import faiss
from sentence_transformers import SentenceTransformer
import json

/opt/miniconda3/envs/dense-sparse/lib/python3.11/site-packages/beir/datasets/data_loader.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Necessary directories

In [2]:
## necessary directories
data_path = "./datasets/fiqa/" ## For the normal docs nfcorpus
bm25_index_path = "indexes/bm25_index/" ## For the inverted index
output_doc_path = "./datasets/docs" ## for the converted docs 
qrel_file = "./datasets/fiqa/qrels/test.tsv" # nfcorpus
splade_index_path = "indexes/splade-index"
splade_path = "./datasets/docs/splade"


#### Models

In [3]:
splade_model = 'naver/splade-cocondenser-ensembledistil' 
model_name = "BAAI/bge-base-en-v1.5"
# model = SentenceTransformer(model_name)


## Load and conver the dataset

In [4]:
## Load the dataset
corpus, queries, qrels = GenericDataLoader(data_path).load()

## Convert the dataset to jsonl
new_data_path = convert_corpus_jsonl(corpus=corpus, output_path=output_doc_path)

  0%|          | 0/57638 [00:00<?, ?it/s]

Saved 57638 documents to ./datasets/docs/corpus.jsonl


## Sparse Retrieval Models - BM25 & SPLADE++

In [5]:

""" This is bm25 or splade model """
# Create the index folder if not exist
os.makedirs(bm25_index_path, exist_ok=True)

bm25 = BM25Retrieval(bm25_index_path) ## We create the index it does not exist
bm25.index(new_data_path) ## Index the data


apr 28, 2026 3:05:33 AM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFORMAZIONI: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


5.856495141983032

In [6]:
splade = Splade(splade_index_path, splade_model)
## Tokenize the corpus
tokens = splade.convert_corpus(f"{new_data_path}/corpus.jsonl", "splade_vectors.jsonl")
## Create the inverted index for splade(or the so called embeddings)
splade.index(f"{new_data_path}/splade")

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [8]:
"""A function to search a query whether in bm25 or splade"""

def sparse_search(algo: str):
    query_list = list(queries.values())
    query_ids = queries.keys()
    
    if algo == "bm25":
        hits, qps = bm25.search(query_list)
    else:
        ## Compute the vectors
        # encoded_queries = []
        # for query in query_list:
        #     vecs = splade.tokenize_and_encode(query)
        #     encoded_queries.append(splade.convert_vec_to_dict(vecs))
                    
        hits, qps = splade.search(query_list)
        
    results = {}
    for q_idx, qid in enumerate(query_ids):
        results[qid] = {}
        for hit in hits[q_idx]:
            # Convert numpy int to int, then map to actual doc ID
            # doc_idx = int(hit['doc_id'])
            results[qid][hit["docid"]] = float(hit['score'])
    print(f"The results of the {algo}:  {results}")
    print(f"The qps of {algo}: {qps}")
    return results, qps
    

In [9]:
"""Then we call the method here"""

results, qps = sparse_search("bm25")
# print(results)

The results of the bm25:  {'PLAIN-2': {'MED-2429': 9.589799880981445, 'MED-10': 9.578900337219238, 'MED-14': 9.434700012207031, 'MED-2428': 8.672200202941895, 'MED-1193': 8.508899688720703, 'MED-2431': 7.764999866485596, 'MED-4827': 7.610899925231934, 'MED-2427': 7.396299839019775, 'MED-3204': 6.92579984664917, 'MED-1887': 6.658599853515625}, 'PLAIN-12': {'MED-4711': 5.451200008392334, 'MED-4262': 4.958399772644043, 'MED-2501': 4.7220001220703125, 'MED-2519': 4.317500114440918, 'MED-1254': 4.286300182342529, 'MED-2311': 4.268899917602539, 'MED-1376': 4.10860013961792, 'MED-4708': 4.1085991859436035, 'MED-2136': 4.105800151824951, 'MED-5042': 4.072999954223633}, 'PLAIN-23': {'MED-2652': 5.8078999519348145, 'MED-2643': 5.7642998695373535, 'MED-2661': 5.7164998054504395, 'MED-2650': 5.55649995803833, 'MED-4451': 5.40500020980835, 'MED-4429': 5.401700019836426, 'MED-2644': 5.115499973297119, 'MED-2658': 5.097899913787842, 'MED-2421': 4.928500175476074, 'MED-5299': 4.900700092315674}, 'PLAI

In [10]:
"""Evaluate the models bm25/splade++"""
metrics = compute_ndcg(qrels, results)

print(metrics)

{'NDCG@1': 0.4257, 'NDCG@10': 0.32178, 'NDCG@100': 0.20768}


In [11]:
"""Then we call the method here"""

results, qps = sparse_search("splade++")
# print(results)

The results of the splade++:  {'PLAIN-2': {'MED-2429': 701.0, 'MED-2431': 675.0, 'MED-4827': 574.0, 'MED-10': 570.0, 'MED-14': 570.0, 'MED-1732': 552.0, 'MED-2427': 551.0, 'MED-2440': 551.0, 'MED-2434': 528.0, 'MED-2122': 523.0}, 'PLAIN-12': {'MED-4711': 431.0, 'MED-2603': 309.0, 'MED-2501': 307.0, 'MED-4113': 263.0, 'MED-3317': 255.0, 'MED-1116': 248.0, 'MED-2028': 248.0, 'MED-3706': 248.0, 'MED-4114': 248.0, 'MED-4115': 248.0}, 'PLAIN-23': {'MED-1961': 436.0, 'MED-1169': 421.0, 'MED-2661': 402.0, 'MED-118': 393.0, 'MED-2651': 393.0, 'MED-2658': 375.0, 'MED-2652': 360.0, 'MED-2495': 355.0, 'MED-4505': 354.0, 'MED-1271': 339.0}, 'PLAIN-33': {'MED-2721': 588.0, 'MED-4763': 545.0, 'MED-3058': 501.0, 'MED-2717': 499.0, 'MED-1803': 495.0, 'MED-2722': 493.0, 'MED-2723': 489.0, 'MED-5006': 486.0, 'MED-1468': 472.0, 'MED-1797': 467.0}, 'PLAIN-44': {'MED-2811': 516.0, 'MED-2817': 506.0, 'MED-2831': 504.0, 'MED-1948': 495.0, 'MED-2814': 493.0, 'MED-2782': 489.0, 'MED-2821': 489.0, 'MED-1112': 4

In [12]:
"""Evaluate the models bm25/splade++"""
metrics = compute_ndcg(qrels, results)

print(metrics)

{'NDCG@1': 0.43344, 'NDCG@10': 0.31908, 'NDCG@100': 0.20898}


# Dense Retrieval Models - Flat & HNSW Index

In [6]:
"""Let's create the embeddings first"""
embeddings = embed_model(model_name, f"{data_path}/corpus.jsonl")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
flat = FlatIndex(embeddings, model_name)
hnsw = HNSW(embeddings, model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
print(f"Embeddings shape: {embeddings.shape}")
print(f"Memory (MB): {embeddings.nbytes / 1024 / 1024}")

Embeddings shape: (3633, 768)
Memory (MB): 10.6435546875


#### Build indexes

In [9]:
flat.build_index()

0.004812002182006836

In [10]:
hnsw.build_index()

0.11538815498352051

## Querying Both Flat & HNSW Index

#### ON-THE-FLY METHOD - computing query embedding on each search

In [13]:
"""A method to compute the query and qps for dense algorithms
for the on-the-fly method"""

def dense_search_on_the_fly(algo: str):
    doc_id_mapping = {}
    with open(f"{data_path}/corpus.jsonl", "r") as f:
        for idx, line in enumerate(f):
            doc = json.loads(line)
            doc_id_mapping[idx] = doc['_id']  # Map index to document ID
            
    query_list = list(queries.values())
    query_ids = list(queries.keys())
    
    if algo == "flat":
        hits, qps = flat.search(query_list)
    else:
        hits, qps = hnsw.search(query_list)
    
    results = {}
    for q_idx, qid in enumerate(query_ids):
        results[qid] = {}
        for hit in hits[q_idx]:
            # Convert numpy int to int, then map to actual doc ID
            doc_idx = int(hit['doc_id'])
            actual_doc_id = doc_id_mapping[doc_idx]
            results[qid][actual_doc_id] = float(hit['score'])
    return results, qps   

In [14]:
"""Call the method here..."""
"""Evaluate Flat - ON-THE-FLY"""

results, qps = dense_search_on_the_fly("flat")
print(f"QPS - ON THE FLY: {qps} Query per second")
print(f"results {results}")

QPS - ON THE FLY: 42866.73180610049 Query per second
results {'PLAIN-2': {'MED-2429': 0.8610804080963135, 'MED-10': 0.8562970757484436, 'MED-14': 0.85297030210495, 'MED-2431': 0.8503385782241821, 'MED-2428': 0.752159059047699, 'MED-2436': 0.7460638880729675, 'MED-4827': 0.7448846697807312, 'MED-2439': 0.7440788149833679, 'MED-5352': 0.7382782101631165, 'MED-3840': 0.7270351648330688}, 'PLAIN-12': {'MED-2512': 0.7218249440193176, 'MED-2325': 0.7194256782531738, 'MED-2510': 0.7140563130378723, 'MED-2502': 0.7135813236236572, 'MED-2514': 0.7106227278709412, 'MED-2501': 0.710583508014679, 'MED-2520': 0.709314227104187, 'MED-2517': 0.7079285383224487, 'MED-4697': 0.7078580856323242, 'MED-4545': 0.6977493166923523}, 'PLAIN-23': {'MED-2654': 0.7235013246536255, 'MED-2651': 0.721179187297821, 'MED-118': 0.721179187297821, 'MED-1961': 0.711181640625, 'MED-4726': 0.7110195755958557, 'MED-1164': 0.7075856328010559, 'MED-2658': 0.704535186290741, 'MED-4168': 0.703189492225647, 'MED-2494': 0.701131

In [15]:
metrics = compute_ndcg(qrels, results)
print(metrics)

{'NDCG@1': 0.45666, 'NDCG@10': 0.36574, 'NDCG@100': 0.23736}


In [16]:
"""Call the method here..."""
"""Evaluate HNWS - ON-THE-FLY"""

results, qps = dense_search_on_the_fly("hnsw")
print(f"QPS - ON THE FLY: {qps} Query per second")
print(f"results {results}")

QPS - ON THE FLY: 30052.355634427684 Query per second
results {'PLAIN-2': {'MED-2429': 0.8610802292823792, 'MED-10': 0.8562972545623779, 'MED-14': 0.8529698848724365, 'MED-2431': 0.8503384590148926, 'MED-2428': 0.7521586418151855, 'MED-2436': 0.7460638284683228, 'MED-4827': 0.7448838949203491, 'MED-2439': 0.7440787553787231, 'MED-5352': 0.7382776737213135, 'MED-3840': 0.7270352840423584}, 'PLAIN-12': {'MED-2512': 0.7218250036239624, 'MED-2325': 0.7194254994392395, 'MED-2510': 0.7140555381774902, 'MED-2502': 0.7135813236236572, 'MED-2514': 0.7106223106384277, 'MED-2501': 0.7105837464332581, 'MED-2520': 0.7093141674995422, 'MED-2517': 0.7079289555549622, 'MED-4697': 0.7078587412834167, 'MED-4545': 0.697749674320221}, 'PLAIN-23': {'MED-2654': 0.723501443862915, 'MED-118': 0.7211792469024658, 'MED-2651': 0.7211792469024658, 'MED-1961': 0.7111815810203552, 'MED-4726': 0.7110195159912109, 'MED-1164': 0.7075862884521484, 'MED-2658': 0.7045351266860962, 'MED-4168': 0.703189492225647, 'MED-2494

In [17]:
metrics = compute_ndcg(qrels, results)
print(metrics)

{'NDCG@1': 0.44737, 'NDCG@10': 0.36061, 'NDCG@100': 0.2332}


#### CACHED METHOD - pre-compute query embedding once then use it for each search

In [18]:
""" We encode the data to use for a cached search"""


model = SentenceTransformer(model_name)

query_lists = list(queries.values())
query_vectors_cached = model.encode(
                query_lists, convert_to_numpy=True
                ).astype("float32")
faiss.normalize_L2(query_vectors_cached)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
""" This is the cached search """
def dense_search_cached(algo: str):
    # Load the corpus to get doc_id mapping
    doc_id_mapping = {}
    with open(f"{data_path}/corpus.jsonl", "r") as f:
        for idx, line in enumerate(f):
            doc = json.loads(line)
            doc_id_mapping[idx] = doc['_id']  # Map index to document ID

    query_ids = list(queries.keys())
    
    if algo == "flat":
        hits, qps = flat.search(query_vectors=query_vectors_cached)
    else:
        hits, qps = hnsw.search(query_vectors=query_vectors_cached)

    results = {}
    for q_idx, qid in enumerate(query_ids):
        results[qid] = {}
        for hit in hits[q_idx]:
            # Convert numpy int to int, then map to actual doc ID
            doc_idx = int(hit['doc_id'])
            actual_doc_id = doc_id_mapping[doc_idx]
            results[qid][actual_doc_id] = float(hit['score'])
    return results, qps

In [26]:
"""Call the method here..."""
"""Evaluate Flat OR - CACHED"""

results, qps = dense_search_cached("flat")
print(f"QPS - CACHED: {qps} Query per second")
print(f"results {results}")

QPS - CACHED: 11728.002354672553 Query per second
results {'PLAIN-2': {'MED-2429': 0.8610804080963135, 'MED-10': 0.8562970757484436, 'MED-14': 0.85297030210495, 'MED-2431': 0.8503385782241821, 'MED-2428': 0.752159059047699, 'MED-2436': 0.7460638880729675, 'MED-4827': 0.7448846697807312, 'MED-2439': 0.7440788149833679, 'MED-5352': 0.7382782101631165, 'MED-3840': 0.7270351648330688}, 'PLAIN-12': {'MED-2512': 0.7218249440193176, 'MED-2325': 0.7194256782531738, 'MED-2510': 0.7140563130378723, 'MED-2502': 0.7135813236236572, 'MED-2514': 0.7106227278709412, 'MED-2501': 0.710583508014679, 'MED-2520': 0.709314227104187, 'MED-2517': 0.7079285383224487, 'MED-4697': 0.7078580856323242, 'MED-4545': 0.6977493166923523}, 'PLAIN-23': {'MED-2654': 0.7235013246536255, 'MED-2651': 0.721179187297821, 'MED-118': 0.721179187297821, 'MED-1961': 0.711181640625, 'MED-4726': 0.7110195755958557, 'MED-1164': 0.7075856328010559, 'MED-2658': 0.704535186290741, 'MED-4168': 0.703189492225647, 'MED-2494': 0.701131284

In [27]:
metrics = compute_ndcg(qrels, results)
print(metrics)

{'NDCG@1': 0.45666, 'NDCG@10': 0.36574, 'NDCG@100': 0.23736}


In [29]:
"""Call the method here..."""
"""Evaluate HNSW - CACHED"""

results, qps = dense_search_cached("hnsw")
print(f"QPS - CACHED: {qps} Query per second")
print(f"results {results}")

QPS - CACHED: 20737.1834073167 Query per second
results {'PLAIN-2': {'MED-2429': 0.8610802292823792, 'MED-10': 0.8562972545623779, 'MED-14': 0.8529698848724365, 'MED-2431': 0.8503384590148926, 'MED-2428': 0.7521586418151855, 'MED-2436': 0.7460638284683228, 'MED-4827': 0.7448838949203491, 'MED-2439': 0.7440787553787231, 'MED-5352': 0.7382776737213135, 'MED-3840': 0.7270352840423584}, 'PLAIN-12': {'MED-2512': 0.7218250036239624, 'MED-2325': 0.7194254994392395, 'MED-2510': 0.7140555381774902, 'MED-2502': 0.7135813236236572, 'MED-2514': 0.7106223106384277, 'MED-2501': 0.7105837464332581, 'MED-2520': 0.7093141674995422, 'MED-2517': 0.7079289555549622, 'MED-4697': 0.7078587412834167, 'MED-4545': 0.697749674320221}, 'PLAIN-23': {'MED-2654': 0.723501443862915, 'MED-118': 0.7211792469024658, 'MED-2651': 0.7211792469024658, 'MED-1961': 0.7111815810203552, 'MED-4726': 0.7110195159912109, 'MED-1164': 0.7075862884521484, 'MED-2658': 0.7045351266860962, 'MED-4168': 0.703189492225647, 'MED-2494': 0.7

In [30]:
metrics = compute_ndcg(qrels, results)
print(metrics)

{'NDCG@1': 0.44737, 'NDCG@10': 0.36061, 'NDCG@100': 0.2332}


## Product quantization

In [ ]:
# flat.build_index(quantize=True)

WARNING clustering 3633 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3633 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3633 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3633 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3633 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3633 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3633 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 3633 points to 256 centroids: please provide at least 9984 training points


0.5530858039855957